In [34]:
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import requests
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from sklearn.compose import make_column_selector, ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn import set_config

from xgboost import XGBClassifier
from interpret.glassbox import ExplainableBoostingClassifier
from interpret import show


# Definición de funciones

In [35]:
def obtener_info_valores_unicos(df):
    cat_cols = df.select_dtypes(include=['object']).columns
    soporte_map = {}
    
    for col in cat_cols:
        iloc_col = df.columns.get_loc(col)
        valores_unicos = frozenset(df[col].dropna().unique())
        
        if valores_unicos not in soporte_map:
            soporte_map[valores_unicos] = []
            
        soporte_map[valores_unicos].append(iloc_col)
            
    return pd.DataFrame([
        {"Indices_iloc": indices, "Soporte": list(valores_unicos)}
        for valores_unicos, indices in soporte_map.items()
    ])


In [36]:
def extraccion_columnas(dataset,tabla_soportes, soporte=[[]]):
    columnas = tabla_soportes[tabla_soportes["Soporte"].isin(soporte)]["Indices_iloc"]
    columnas_indices = list({item for sublist in columnas for item in sublist})
    columnas_nombres = dataset.columns[columnas_indices]
    return columnas_nombres

# Definición de los datos

In [37]:
load_dotenv(find_dotenv())

PROJECT_ROOT = Path(find_dotenv()).parent.resolve()
print(f"PROJECT_ROOT set to: {PROJECT_ROOT}")
DATA_DIR = PROJECT_ROOT / 'data'

PROJECT_ROOT set to: C:\Users\rtrin\OneDrive\Escritorio\FORMACION\MASTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025


In [38]:
datos_original=pd.read_csv(DATA_DIR / 'synthea_30k.csv',low_memory=False)
# datos= pd.DataFrame(datos_original['label'],index=datos_original["ptnum"])


In [39]:
datos_SNOMED = pd.read_csv(DATA_DIR / 'synthea_snomed.csv', low_memory=False)
datos_MEDICAMENTOS= pd.read_csv(DATA_DIR / 'synthea_rxnorm.csv',low_memory=False)
datos_LABORATORIO=  pd.read_csv(DATA_DIR / 'synthea_loinc.csv',low_memory=False)
datos_totales = pd.concat([datos_SNOMED,datos_LABORATORIO, datos_MEDICAMENTOS], axis=1)

In [40]:
datos_totales.drop(columns="ptnum",inplace=True)

In [41]:
X=datos_totales.drop(columns="label")
y=datos_totales["label"]

In [42]:
X_train, X_test, y_train, y_test = train_test_split(X,y,stratify=y,random_state=25)

# Datos SNOMED

In [43]:
obtener_info_valores_unicos(datos_SNOMED)

,Indices_iloc,Soporte
0,[220],"[hawaiian, asian, other, black, white, native]"
1,[221],"[s, m]"
2,[222],"[nonhispanic, hispanic]"
3,[223],"[m, f]"
4,[224],[massachusetts]
5,[225],"[30t50, 50t70, gt70]"


In [44]:
col_OH=datos_SNOMED.columns[[220]]
col_OR=datos_SNOMED.columns[[221,222,223,224]]
col_edad=datos_SNOMED.columns[[225]]


## Pipeline para diagnosticos y procedimientos

In [45]:
OR_pipeline = Pipeline([
    ("codif", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1,encoded_missing_value=-2))
])
OH_pipeline = Pipeline([
    ("codif",OneHotEncoder(handle_unknown='ignore',sparse_output=False))])

edad_pipeline = Pipeline([
    ("codif",OrdinalEncoder(categories=[["30t50","50t70","gt70"]],handle_unknown='use_encoded_value', unknown_value=-1,encoded_missing_value=-2))])


preprocesamiento_SNOMED = ColumnTransformer([
    ("OR", OR_pipeline, col_OR),
    ("OH", OH_pipeline, col_OH),
    ("edad", edad_pipeline, col_edad)
], remainder="passthrough", verbose_feature_names_out=False)



# Datos Medicamentos

In [46]:
obtener_info_valores_unicos(datos_MEDICAMENTOS)

,Indices_iloc,Soporte
0,[20],[cloudy]
1,[22],[abnormal]


In [47]:
col_med=datos_MEDICAMENTOS.columns[[20,22]]

## Pipeline para resultados de laboratorio

In [48]:
encoder_pipeline = Pipeline([
    ("codif", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1,encoded_missing_value=-2))
])

preprocesamiento_medicamentos = ColumnTransformer([
    ("encoder", encoder_pipeline,col_med )
], remainder="passthrough", verbose_feature_names_out=False)



# DATOS LABORATORIO

In [49]:
datos_LABORATORIO.drop(columns=["ptnum","label"],inplace=True)

Ahora que tenemos desglosados los datos, hacer el preprocesamiento es mucho más claro, pues es mucho más simple ver los tipos de las variables, cuantos valores, cómo categorizarlas y como imputar los valores faltantes.

## Variables numéricas

In [50]:
datos_LABORATORIO.select_dtypes(exclude="object").info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4221 entries, 0 to 4220
Data columns (total 18 columns):
 #   Column                                                    Non-Null Count  Dtype  
---  ------                                                    --------------  -----  
 0   C reactive protein [Mass/volume] in Serum or Plasma       93 non-null     float64
 1   Red Blood Cell                                            22 non-null     float64
 2   White Blood Cell (Elevated)                               22 non-null     float64
 3   Platelet Count                                            22 non-null     float64
 4   Total Bilirubin (Elevated)                                22 non-null     float64
 5   Fibrin D-dimer FEU [Mass/volume] in Platelet poor plasma  93 non-null     float64
 6   Prothrombin time (PT)                                     93 non-null     float64
 7   Monocytes/100 leukocytes in Blood by Automated count      93 non-null     float64
 8   INR in Platelet po

Como vemos, estas variables numéricas tienen muy pocos datos no nulos, por lo que en este punto tendríamos dos opciones, elimnarlas por completo o discretizarlas, creando una categoría para NAs y otra para cada rango de valores de la variable o directamente estandarizarlas sin tener en cuenta los nulos.

## Variables categóricas

In [51]:
datos_lab_obj=datos_LABORATORIO.select_dtypes(include="object")


In [52]:
tabla_soportes=obtener_info_valores_unicos(datos_lab_obj)
tabla_soportes

,Indices_iloc,Soporte
0,"[0, 1, 2, 3, 4, 6, 7, 8, 9, 10, 12, 13, 14, 15...","[normal, abnormal]"
1,[5],"[former, never]"
2,"[11, 18, 20, 21, 33, 36, 42]",[normal]
3,[28],"[3+, 2+, 1+]"
4,[30],[2+]
5,[31],[True]
6,[32],"[brown, reddish]"
7,[34],"[negative, positive]"
8,[35],[negative]
9,"[38, 49]",[abnormal]


La mayor parte de las variables de laboratorio toma valores "normal" o "abnormal". Sin embargo, algunas sueltas pueden tomar otros valores. Diferenciaremos estas por grupos de variables para aplicar el *label encoder*

In [53]:
columnas_normal_abnormal = extraccion_columnas(datos_lab_obj,tabla_soportes,[["normal"],["normal","abnormal"],["abnormal"]])
columnas_former_never = extraccion_columnas(datos_lab_obj,tabla_soportes,[["former","never"],["never","former"]])
columnas_nums= extraccion_columnas(datos_lab_obj,tabla_soportes,[["2+","3+", "1+"],["2+"],["3+","2+", "1+"]])
columnas_brown_reddish = extraccion_columnas(datos_lab_obj,tabla_soportes,[["brown","reddish"]])
columnas_negative_positive= extraccion_columnas(datos_lab_obj,tabla_soportes,[["negative"],["negative","positive"],["positive"]])
columnas_false_true= extraccion_columnas(datos_lab_obj,tabla_soportes,[[False,True],[True]])

## Pipeline para resultados de laboratorio

In [54]:
AN_pipeline = Pipeline([
    ("codif", OrdinalEncoder(categories=[["abnormal", "normal"]]* len(columnas_normal_abnormal),handle_unknown='use_encoded_value', unknown_value=-1,encoded_missing_value=-2))
])
FT_pipeline = Pipeline([
    ("codif", OrdinalEncoder(categories=[[False,True]] * len(columnas_false_true),handle_unknown='use_encoded_value', unknown_value=-1,encoded_missing_value=-2))
])
NP_pipeline = Pipeline([
    ("codif", OrdinalEncoder(categories=[["negative","positive"]]* len(columnas_negative_positive),handle_unknown='use_encoded_value', unknown_value=-1,encoded_missing_value=-2))
])
NUMS_pipeline = Pipeline([
    ("codif", OrdinalEncoder(categories=[["1+","2+","3+"]]* len(columnas_nums),handle_unknown='use_encoded_value', unknown_value=-1,encoded_missing_value=-2))
])
NF_pipeline = Pipeline([
    ("codif", OrdinalEncoder(categories=[["never","former"]]* len(columnas_former_never),handle_unknown='use_encoded_value', unknown_value=-1,encoded_missing_value=-2))
])
BR_pipeline = Pipeline([
    ("codif", OrdinalEncoder(categories=[["brown","reddish"]]* len(columnas_brown_reddish),handle_unknown='use_encoded_value', unknown_value=-1,encoded_missing_value=-2))
])


num_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

preprocesamiento_lab = ColumnTransformer([
    ("AN", AN_pipeline, columnas_normal_abnormal),
    ("FT", FT_pipeline, columnas_false_true),
    ("NP", NP_pipeline, columnas_negative_positive),
    ("NF", NF_pipeline, columnas_former_never),
    ("NUMS", NUMS_pipeline, columnas_nums),
    ("BR", BR_pipeline, columnas_brown_reddish),

    ("num", num_pipeline,make_column_selector(dtype_exclude="object"))
], remainder="passthrough", verbose_feature_names_out=False)



In [55]:
preprocesamiento_lab.fit_transform(datos_lab_obj,y)

,Low Density Lipoprotein Cholesterol,High Density Lipoprotein Cholesterol,Total Cholesterol,Triglycerides,Body Mass Index,Diastolic Blood Pressure,Systolic Blood Pressure,Heart rate,Respiratory rate,Erythrocyte distribution width [Entitic volume] by Automated count,...,Thyrotropin [Units/volume] in Serum or Plasma,Thyroxine (T4) free [Mass/volume] in Serum or Plasma,SARS-CoV-2 RNA Pnl Resp NAA+probe,Bilirubin.total [Presence] in Urine by Test strip,Hemoglobin [Presence] in Urine by Test strip,Leukocyte esterase [Presence] in Urine by Test strip,Tobacco smoking status NHIS,Protein [Presence] in Urine by Test strip,Glucose [Presence] in Urine by Test strip,Color of Urine
0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,...,-1.0,-1.0,-1.0,1.0,0.0,0.0,1.0,2.0,1.0,0.0
1,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,...,-1.0,-1.0,-1.0,1.0,1.0,0.0,0.0,2.0,1.0,1.0
2,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,...,-1.0,-1.0,-1.0,1.0,1.0,0.0,0.0,2.0,1.0,1.0
3,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,...,-1.0,-1.0,-1.0,1.0,1.0,0.0,1.0,2.0,1.0,1.0
4,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,...,-1.0,-1.0,-1.0,1.0,1.0,0.0,0.0,2.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4216,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0,-1.0,-1.0,-1.0
4217,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0,-1.0,-1.0,-1.0
4218,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,-1.0,-1.0,-1.0
4219,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0,-1.0,-1.0,-1.0


# Pipeline General

In [56]:
columnas_snomed = datos_SNOMED.columns.tolist()
columnas_medicamentos = datos_MEDICAMENTOS.columns.tolist()
columnas_laboratorio = datos_LABORATORIO.columns.tolist()

In [61]:
preprocesamiento = ColumnTransformer(
    transformers=[
        ("snomed",preprocesamiento_SNOMED,columnas_snomed),
        ("medicamentos",preprocesamiento_medicamentos,columnas_medicamentos),
        ("lab", preprocesamiento_lab, columnas_laboratorio)],
    remainder="passthrough", 
    verbose_feature_names_out=False
)

modelo_xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=25
)
set_config(transform_output="default")

pipeline_final = Pipeline(steps=[
    ('preprocesamiento', preprocesamiento),
    ('modelo_xgb', modelo_xgb)               
])

In [64]:
modelo=pipeline_final.fit(X_train,y_train)

In [65]:
y_pred=modelo.predict(X_test)


In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       766
           1       0.99      0.93      0.96       290

    accuracy                           0.98      1056
   macro avg       0.98      0.97      0.97      1056
weighted avg       0.98      0.98      0.98      1056



In [ ]:
preprocesador = pipeline_final.named_steps['preprocesamiento']
modelo_xgb = pipeline_final.named_steps['modelo_xgb']

nombres_variables = preprocesador.get_feature_names_out()

importancias = modelo_xgb.feature_importances_

KeyError: 'modelo_xgb'

# Explainable Boosting Machines

In [ ]:
ebm = ExplainableBoostingClassifier(
    random_state=25,
    interactions=10 
)
set_config(transform_output="pandas")


pipeline_ebm = Pipeline(steps=[
    ('preprocesamiento', preprocesamiento), 
    ('modelo_ebm', ebm)                
])

In [ ]:
modelo_ebm=pipeline_ebm.fit(X_train,y_train)

c:\Users\rtrin\OneDrive\Escritorio\FORMACION\MASTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025\.pixi\envs\default\Lib\site-packages\interpret\glassbox\_ebm\_ebm.py:871: UserWarning: Missing values detected. Our visualizations do not currently display missing values. To retain the glassbox nature of the model you need to either set the missing values to an extreme value like -1000 that will be visible on the graphs, or manually examine the missing value score in ebm.term_scores_[term_index][0]
  warn(


In [ ]:
ebm=modelo_ebm.named_steps["modelo_ebm"]

In [ ]:
show(ebm.explain_global())

<!-- http://127.0.0.1:7001/2641758022736/ -->